# Transformer 架构源码解析

**请输入文本，重量级别的架构，之后的很多经典之作(比如 ViT)相当一部分都是基于它，可以说是祖师爷级别的了**

Transformer 的核心架构如下图所示:

<div align="center">
    <img src="./imgs/Transformer_architecture.png" alt="Transformer Architecture" style="width: 350px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/398039366)

[Transformer 论文](https://arxiv.org/pdf/1706.03762)

[Transformer 模型 Pytorch 实现](https://github.com/harvardnlp/annotated-transformer)



## Attention

### Attention 机制
注意力的计算过程如下图所示，其中 Q、K、V 是输入 X 通过三个不同的线性层得到的，线性层的参数可学习，我将在后面结合源码的实现进行解释:

<div align="center">
    <img src="./imgs/Scaled Dot-Product Attention.png" alt="Scaled Dot-Product Attention" style="width: 200px; height: auto;">
</div>

公式表现如下:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

其中$d_k$是每个注意力头的维度，在单头注意力中等于嵌入维度，至于为什么要除以缩放因子$\sqrt{d_k}$，是因为要防止点积过大导致梯度消失的问题

### softmax
softmax 函数的公式如下，假设有一个向量$z = [z_1, \ldots, z_n]$:

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}
$$

它的作用在于，把 z 里的值全部变成0~1之间的一个数，且保证**它们之间的和为1**，在分类任务中，softmax 之后的向量的每一个值，表示属于该类别的概率，如 z[0] 表示属于第0个类别的概率

### $QK^T$ 的含义

假设 Q、K、V 的维度都是 (seq_len, dim)，那么经过这一层的运算之后得到的结果就是 (seq_len, seq_len)，即表示两两 token 之间的**注意力分数**，比如位置[0][2]，就表示第0个 token 要给予第2个 token 多少关注度


In [156]:
import torch 
import torch.nn as nn 
import numpy as np 
import math 
import copy
from torch.nn.functional import log_softmax, pad


def clones(module, N):
    "Produce N identical layers(制作N个相同的层)."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


In [157]:
def attention(query, key, value, mask=None, dropout=None, vis=False):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)  # 获取d_k
    # -> [seq_len, seq_len]
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # 应用mask时，mask为0的位置设置为一个极小值
        # 后面我会对mask进行解释
        scores = scores.masked_fill(mask == 0, -1e9)
    
    p_attn = scores.softmax(dim=-1)

    if dropout is not None:
        p_attn = dropout(p_attn)

    # -> [seq_len, dim]
    output = torch.matmul(p_attn, value)

    if vis:
        print(f'[attention inner] attention scores size: {p_attn.size()}')
        print(f'[attention inner] attention scores row sum: {p_attn[0, 0].sum()}')

    return output, p_attn

In [158]:
batch = 2
seq_len = 4
d_k = 8
d_v = 8
query = torch.randn(batch, seq_len, d_k)
key = torch.randn(batch, seq_len, d_k)
value = torch.randn(batch, seq_len, d_v)

# 我们可以看到输入和输出的形状都是一致的，且注意力权重行和为1
print(f'Before attention, QKV size: {query.size()}')
output, attn_weights = attention(query, key, value, vis=True)
print(f'After attention, output size: {output.size()}')

Before attention, QKV size: torch.Size([2, 4, 8])
[attention inner] attention scores size: torch.Size([2, 4, 4])
[attention inner] attention scores row sum: 1.0000001192092896
After attention, output size: torch.Size([2, 4, 8])


## Multi-Head Attention
简单来说，多头注意力就是使用多个注意力头作用，之后把所有头的输出结果进行拼接后通过一层线性层得到最后的输出，使用多个头是为了学习不同的关系，比如头0学习主语和动词的关系，头1学习形容词和名词的关系，如下图所示:

<div align="center">
    <img src="./imgs/Multi-Head Attention.png" alt="Multi-Head Attention" style="width: 200px; height: auto;">
</div>

假设 dim 为512，注意力头数 num_heads 为8，那么每个头的维度 $d_k$ 就是 512/8 = 64，那么每一个头作用之后的维度是 (seq_len, d_k)，在通道维度拼接之后就能得到 (seq_len, dim)，在经过一层线性层作用就得到了多头注意力的输出了，如果要用公式总结一下，就如下所示:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O

\\

\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

至于为什么我在第二个公式中单独写出Q、K、V，是因为他们可能并不直接来自于同一个序列，也就是后面会介绍的交叉注意力，下文中我会进行介绍



In [159]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        
        # We assume d_v always equals d_k
        self.d_k = d_model // h  # 每个注意力头的维度
        self.h = h  # 注意力头数
        self.linears = clones(nn.Linear(d_model, d_model), 4)  # Q、K、V、output的线性层
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None, vis=False):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        # --------------------------------------------------------------------------
        # 1) Do all the linear projections in batch from d_model => h x d_k
        # --------------------------------------------------------------------------
        """
        重点讲一下这几行代码，非常简洁漂亮的代码完成了投影和分头的操作:
        - for lin, x in zip(self.linears, (query, key, value)):
            就是组合成了linear0 -> query / linear1 -> key / linear2 -> value
            三个组合，让Q、K、V对应的线性投影层分别作用于他们

        - lin(x).view(nbatches, -1, self.h, self.d_k):
            线性投影之后重塑张量形状: [nbatches, seq_len, dim] -> [nbatches, seq_len, num_heads, d_k]

        - .transpose(1, 2):
            交换seq_len, num_heads这两个维度，即[nbatches, seq_len, num_heads, d_k] -> [nbatches, num_heads, seq_len, d_k]
            因为每个头需要[seq_len, d_k]这样的期望输入
        """
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]  # -> [nbatches, num_heads, seq_len, d_k]
        
        if vis:
            print(f'[MultiHeadAttention inner] After divide for heads, QKV size: {query.size()}')

        # --------------------------------------------------------------------------
        # 2) Apply attention on all the projected vectors in batch.
        # --------------------------------------------------------------------------
        """
        对所有头并行计算注意力
        """
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )  # -> [nbatches, num_heads, seq_len, d_k] / [nbatches, num_heads, seq_len, seq_len]
        
        if vis:
            print(f'[MultiHeadAttention inner] After attention for heads, output1 size: {x.size()}')
            print(f'[MultiHeadAttention inner] After attention for heads, QK^T size: {self.attn.size()}')

        # --------------------------------------------------------------------------
        # 3) "Concat" using a view and apply a final linear.
        # --------------------------------------------------------------------------
        """
        最后的多头拼接和投影输出(return)部分:
            - x.transpose(1, 2):
                [nbatches, num_heads, seq_len, d_k] -> [nbatches, seq_len, num_heads, d_k]

            - .contiguous():
                确保内存连续，从而进行view操作

            - .view(nbatches, -1, self.h * self.d_k):
                [nbatches, seq_len, num_heads, d_k] -> [nbatches, seq_len, dim]
        """
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        
        if vis:
            print(f'[MultiHeadAttention inner] After concat for heads, output2 size: {x.size()}')

        return self.linears[-1](x)  # -> [nbatches, seq_len, dim]
    

In [160]:
num_heads = 4
dim = 512
batch_size = 114
seq_len = 514
MHA = MultiHeadedAttention(num_heads, dim)
q = k = v = torch.randn(batch_size, seq_len, dim)

print(f'Before Multi Head Attention, QKV size: {q.size()}')
MHA_output = MHA(q, k, v, vis=True)
print(f'After Multi Head Attention, output size: {MHA_output.size()}')

Before Multi Head Attention, QKV size: torch.Size([114, 514, 512])
[MultiHeadAttention inner] After divide for heads, QKV size: torch.Size([114, 4, 514, 128])
[MultiHeadAttention inner] After attention for heads, output1 size: torch.Size([114, 4, 514, 128])
[MultiHeadAttention inner] After attention for heads, QK^T size: torch.Size([114, 4, 514, 514])
[MultiHeadAttention inner] After concat for heads, output2 size: torch.Size([114, 514, 512])
After Multi Head Attention, output size: torch.Size([114, 514, 512])


## Positionwise Feed-Forward Network
就是经过注意力层之后的一个 MLP 网络，比较好理解，同样地，输入维度等于输出维度 (seq_len, dim)

In [161]:

class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))
    

In [162]:
PFF = PositionwiseFeedForward(dim, 4*dim)

print(f'Before FFN, x size: {MHA_output.size()}')
PFF_out = PFF(MHA_output)
print(f'Before FFN, x size: {PFF_out.size()}')


Before FFN, x size: torch.Size([114, 514, 512])
Before FFN, x size: torch.Size([114, 514, 512])


## Embedding

### Token Embedding
就是对每一个 token 查表找到属于它的嵌入，这个表 Embedding 的大小是 (vocabulary_size, dim)，vocabulary_size 就是整个词表的大小，比如对于第0个 token，它的 Token Embedding 就是 Embedding[0]

In [163]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        # 进行了放缩，避免梯度消失/爆炸
        return self.lut(x) * math.sqrt(self.d_model)

In [164]:
EB = Embeddings(12, 1000)
token_ids =  torch.randint(0, 1000, (1, 1))

print(f'Before Embedding: {token_ids}')
seq = EB(token_ids)
print(f'After Embedding: {seq}')


Before Embedding: tensor([[54]])
After Embedding: tensor([[[ -0.3748, -10.6118,  -1.5866,   2.9236,  -3.5615,  -1.3697,   0.3890,
            0.5628,  -2.0517,   1.9608,  -2.2513,  -0.6750]]],
       grad_fn=<MulBackward0>)


### Positional Encoding
为了考虑句子中词的位置信息，需要通过位置编码把每个词的信息嵌入到 Embedding 中，有两种实现: 固定编码(如正余弦)和可学习编码，这里讲解正余弦固定编码，公式如下:

$$
\text{PE}_{pos, 2i} = \sin\left(pos/10000^{2i/d_{model}}\right)

\\

\text{PE}_{pos, 2i+1} = \cos\left(pos/10000^{2i/d_{model}}\right)
$$

**注意**，在代码中，是通过下面的式子进行计算的，因为能够简化计算、提升数值稳定性，并且二者等价(**代码里的 log 指的是数学上的 ln**，不要搞混):

$$
\text{PE}_{pos, 2i} = \sin\left(pos \times \exp\left(-2i \times \log(10000) / d_{model}\right)\right)

\\

\text{PE}_{pos, 2i+1} = \cos\left(pos \times \exp\left(-2i \times \log(10000) / d_{model}\right)\right)
$$


In [165]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 直接把位置编码加到Embedding中
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

In [166]:
PE = PositionalEncoding(12, 0.1)

print(f'Before Positional Embedding: {seq}')
seq = PE(seq)
print(f'After Positional Embedding: {seq}')


Before Positional Embedding: tensor([[[ -0.3748, -10.6118,  -1.5866,   2.9236,  -3.5615,  -1.3697,   0.3890,
            0.5628,  -2.0517,   1.9608,  -2.2513,  -0.6750]]],
       grad_fn=<MulBackward0>)
After Positional Embedding: tensor([[[ -0.0000, -10.6797,  -0.0000,   4.3596,  -0.0000,  -0.4108,   0.4322,
            1.7364,  -2.2797,   3.2898,  -0.0000,   0.3611]]],
       grad_fn=<MulBackward0>)


## Layer Norm
层归一化，目的是让神经网络每一层的输出数值分布更稳定，解决训练过程中梯度消失/爆炸的问题，公式如下，其中 $\gamma$、$\beta$ 可学习:

$$
\text{LayerNorm(x)} = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

$\mu$、$\sigma^2$ 为所有特征(列)的均值和方差，$\epsilon$ 是一个极小的数，防止除零


In [167]:

class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        # x: [bs, seq_len, dim]
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        
        # a_2 / b_2发生了广播: [dim] -> [bs, seq_len, dim]
        # 在批次和序列维度上进行了复制
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2
    

In [168]:
LN = LayerNorm(12)

print(f'Before Layer Norm: {seq}')
seq = LN(seq)
print(f'After Layer Norm: {seq}')


Before Layer Norm: tensor([[[ -0.0000, -10.6797,  -0.0000,   4.3596,  -0.0000,  -0.4108,   0.4322,
            1.7364,  -2.2797,   3.2898,  -0.0000,   0.3611]]],
       grad_fn=<MulBackward0>)
After Layer Norm: tensor([[[ 0.0716, -2.8039,  0.0716,  1.2454,  0.0716, -0.0390,  0.1880,
           0.5391, -0.5422,  0.9574,  0.0716,  0.1688]]],
       grad_fn=<AddBackward0>)


## Sub Layer Connection
加和层，就是 Add & Norm 操作，很容易，输入输出维度依旧一致

In [169]:

class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        # sublayer就是注意力层或者前馈层
        return x + self.dropout(sublayer(self.norm(x)))
    

## Encoder & Decoder
你看到了这里就已经掌握的差不都了，最终的编码器、解码器就是依据架构图，把上面的组件进行一定的组装

### Encoder Layer
Encoder 包含自注意力层和前馈层

In [170]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        
        self.self_attn = self_attn  # 自注意力
        self.feed_forward = feed_forward  # 前馈
        self.sublayer = clones(SublayerConnection(size, dropout), 2)  # Add & Norm
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        # 自注意力+前馈
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        
        return self.sublayer[1](x, self.feed_forward)


In [171]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)
    

### Decoder Layer
按照架构图，解码层由掩码多头自注意力、编码器-解码器注意力(Q 来自节码器，K、V 来自编码器)、前馈网络

In [172]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        
        self.size = size
        self.self_attn = self_attn  # 掩码多头自注意力
        self.src_attn = src_attn  # 编码器-解码器注意力
        self.feed_forward = feed_forward  # 前馈网络
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory  # 来自编码器
        # tgt_mask用于遮住未来的词，防止模型提前“偷看答案”
        # src_mask用于遮住padding(补0，让句子长度都一样)，排除无意义的padding
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        
        return self.sublayer[2](x, self.feed_forward)
    

In [173]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."
    def __init__(self, layer, N):
        super(Decoder, self).__init__()

        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        
        return self.norm(x)

###  EncoderDecoder
最终的编码器-解码器架构

他是这样工作的: 假设已经有了输入语言[I Like playing GS]，那么他就会从第一个 token 开始，循环解码: 

[\<start>] -> [\<start> 我] -> [\<start>我 爱] -> [\<start>我 爱 玩] -> [\<start>我 爱 玩 GS] -> [\<start> 我 爱 玩 GS <end>]

也就是说，从只有一个 start 标志的序列开始输入，每次输出得到一个新的词，把它加入到序列末尾，再把新得到的序列输入到解码器中，直到输出 end 


In [174]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed  # 源语言嵌入
        self.tgt_embed = tgt_embed  # 目标语言嵌入
        self.generator = generator  # 最终的Linear + Softmax生成词概率

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
    

### 使用示例
这里我直接使用 github 上的[源码](https://github.com/harvardnlp/annotated-transformer)来进行演示

In [175]:

def make_model(
    src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1
):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),  # 源语言嵌入
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),  # 源语言嵌入
        Generator(d_model, tgt_vocab),
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    # 用Xavier初始化，让初始让权重分布均匀，防止梯度消失/爆炸
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    
    return model


def subsequent_mask(size):
    "Mask out subsequent positions."
    # 生成下三角矩阵掩码，让解码器在生成第 n 个词时，
    # 只能看到前面的词，看不到后面的词
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.uint8
    )
    
    return subsequent_mask == 0


In [176]:
def inference_test():
    test_model = make_model(11, 11, 2)
    test_model.eval()
    # 源语言句子和mask(全为1就是都不mask)
    src = torch.LongTensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
    src_mask = torch.ones(1, 1, 10)

    memory = test_model.encode(src, src_mask)  # 源语言句子进行编码理解
    ys = torch.zeros(1, 1).type_as(src)  # 从start(即0)开始翻译

    for i in range(9):  # 生成九个词
        out = test_model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )  # -> [1, 当前句子长度, 512]
        prob = test_model.generator(out[:, -1])  # 只预测最后一个词，即新词
        _, next_word = torch.max(prob, dim=1)  # 取概率最高的预测词
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )  # 拼接新词，之后继续解码
        print(f'step {i+1}: {ys}')

    print("Example Untrained Model Prediction:", ys)

def run_tests():
    for _ in range(1):
        inference_test()

def show_example(fn, args=[]):
    if __name__ == "__main__" and True:
        return fn(*args)

show_example(run_tests)


step 1: tensor([[0, 6]])
step 2: tensor([[0, 6, 4]])
step 3: tensor([[0, 6, 4, 1]])
step 4: tensor([[ 0,  6,  4,  1, 10]])
step 5: tensor([[ 0,  6,  4,  1, 10,  1]])
step 6: tensor([[ 0,  6,  4,  1, 10,  1,  1]])
step 7: tensor([[ 0,  6,  4,  1, 10,  1,  1,  1]])
step 8: tensor([[ 0,  6,  4,  1, 10,  1,  1,  1,  1]])
step 9: tensor([[ 0,  6,  4,  1, 10,  1,  1,  1,  1,  1]])
Example Untrained Model Prediction: tensor([[ 0,  6,  4,  1, 10,  1,  1,  1,  1,  1]])
